In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/abdelrahmanmalekg/taxi-spark-project/sample.csv
/kaggle/input/datasets/abdelrahmanmalekg/taxi-spark-project/nyc-boroughs.geojson


In [3]:
# writing all the imports needed during this note book 😉
import sys
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.functions import explode
from shapely.geometry import shape, Point
from pyspark.sql.window import Window

# Starting the SparkSession
spark = (
    SparkSession
    .builder
    .appName("NYC_Taxi_Spark_Project")
    .master("local[4]")
    .config("spark.dynamicAllocation.enabled", "false")
    .config("spark.sql.adaptive.enabled", "false")
    .getOrCreate()
        )


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/06 11:26:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
# Knowing the input files paths
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/abdelrahmanmalekg/taxi-spark-project/sample.csv
/kaggle/input/datasets/abdelrahmanmalekg/taxi-spark-project/nyc-boroughs.geojson


In [5]:
# Reading the initial DataFrame
taxiDF = (
    spark
    .read
    .option("inferSchema", "true")
    .option("header", True)
    .csv("/kaggle/input/datasets/abdelrahmanmalekg/taxi-spark-project/sample.csv")
)
# knowing number of partitions of my initial DataFrame
print("Partitions = " + str(taxiDF.rdd.getNumPartitions()))
# Knowing my initial DataFrame number of Records
print("Record Count = " + str(taxiDF.count()))

Partitions = 4
Record Count = 99999


In [6]:
# Modifying the Suffle Partitions from 200 -> 8 
spark.conf.set("spark.sql.shuffle.partitions", "8")

In [40]:
# Seeing my DataFrame Schema
taxiDF.printSchema()

root
 |-- medallion: string (nullable = true)
 |-- hack_license: string (nullable = true)
 |-- vendor_id: string (nullable = true)
 |-- rate_code: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_time_in_secs: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- pickup_borough: string (nullable = true)
 |-- dropoff_borough: string (nullable = true)
 |-- pickup_ts: long (nullable = true)
 |-- dropoff_ts: long (nullable = true)
 |-- previous_dropoff_ts: long (nullable = true)
 |-- idle_time: long (nullable = true)
 |-- idle_time_clean: long (nullable = true)



In [8]:
# Reading the GeoJson file
geoJsonDF = (
    spark
    .read
    .option("inferSchema", "true")
    .option("multiline", "true")
    .json("/kaggle/input/datasets/abdelrahmanmalekg/taxi-spark-project/nyc-boroughs.geojson")
)

print(("Partitions = " + str(geoJsonDF.rdd.getNumPartitions())))
print("Record Count = " + str(geoJsonDF.count()))

Partitions = 1
Record Count = 1


In [9]:
geoJsonDF.printSchema()

root
 |-- features: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- geometry: struct (nullable = true)
 |    |    |    |-- coordinates: array (nullable = true)
 |    |    |    |    |-- element: array (containsNull = true)
 |    |    |    |    |    |-- element: array (containsNull = true)
 |    |    |    |    |    |    |-- element: double (containsNull = true)
 |    |    |    |-- type: string (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- properties: struct (nullable = true)
 |    |    |    |-- @id: string (nullable = true)
 |    |    |    |-- borough: string (nullable = true)
 |    |    |    |-- boroughCode: long (nullable = true)
 |    |    |-- type: string (nullable = true)
 |-- type: string (nullable = true)



In [10]:
geoJsonDF.show()

+--------------------+-----------------+
|            features|             type|
+--------------------+-----------------+
|[{{[[[-74.0505080...|FeatureCollection|
+--------------------+-----------------+



In [11]:
# Explosing the features list to be able to access each feature as a seperate row
geoJsonDF = geoJsonDF.select(explode("features").alias("feature"))

In [12]:
# Reading the important data from geoJson
geoJsonDF = geoJsonDF.select(
    "feature.properties.borough",
    "feature.properties.boroughCode",
    "feature.geometry"
)

In [13]:
geoJsonDF.show()

+-------------+-----------+--------------------+
|      borough|boroughCode|            geometry|
+-------------+-----------+--------------------+
|Staten Island|          5|{[[[-74.050508064...|
|Staten Island|          5|{[[[-74.053140368...|
|Staten Island|          5|{[[[-74.159456024...|
|Staten Island|          5|{[[[-74.082212729...|
|       Queens|          4|{[[[-73.836682741...|
|       Queens|          4|{[[[-73.813396652...|
|       Queens|          4|{[[[-73.827182821...|
|       Queens|          4|{[[[-73.826074726...|
|       Queens|          4|{[[[-73.832442073...|
|       Queens|          4|{[[[-73.794201726...|
|       Queens|          4|{[[[-73.805097201...|
|       Queens|          4|{[[[-73.804991988...|
|       Queens|          4|{[[[-73.739558564...|
|       Queens|          4|{[[[-73.739443808...|
|       Queens|          4|{[[[-73.790549485...|
|       Queens|          4|{[[[-73.766708277...|
|       Queens|          4|{[[[-73.797835333...|
|       Queens|     

In [14]:
# Collecting this Dataframe to make it python list to apply some functions on it
geoJsonDF = geoJsonDF.collect()

In [15]:
# Ingesting the data from spark rows to a python list
geoJsonDF = [
    {
        "boroughCode": row["boroughCode"],
        "borough": row["borough"],
        "geometry": row["geometry"].asDict()
    }
    for row in geoJsonDF
]

In [16]:
# sorting the geoJson list to be faster to be used, so the frequently used borough are at the top 
geoJsonDF = sorted(
    geoJsonDF, 
    key = lambda x: (x["boroughCode"], -shape(x["geometry"]).area)
)

In [17]:
# Broadcasting the python list to make at each node for further processing
geo_broadcast = spark.sparkContext.broadcast(geoJsonDF)

In [41]:
# Defining the function that will let us know which borough we are in
def get_borough(lon, lat):
    point = Point(lon, lat)

    for row in geo_broadcast.value:
        polygon = shape(row["geometry"])
        if polygon.contains(point):
            return row["borough"]

    return None

In [19]:
# Transforming the python udf to spark udf 
get_borough_udf = udf(get_borough, StringType())

In [20]:
taxiDF.printSchema()

root
 |-- medallion: string (nullable = true)
 |-- hack_license: string (nullable = true)
 |-- vendor_id: string (nullable = true)
 |-- rate_code: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_time_in_secs: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)



In [21]:
# Adding the pickup_borough column
taxiDF = taxiDF.withColumn(
    "pickup_borough",
    get_borough_udf("pickup_longitude", "pickup_latitude")
)

In [42]:
# Adding the dropoff_borough column

taxiDF = taxiDF.withColumn(
    "dropoff_borough",
    get_borough_udf("dropoff_longitude", "dropoff_latitude")
)

In [23]:
taxiDF.show(5)

+--------------------+--------------------+---------+---------+------------------+-------------------+-------------------+---------------+-----------------+-------------+----------------+---------------+-----------------+----------------+--------------+---------------+
|           medallion|        hack_license|vendor_id|rate_code|store_and_fwd_flag|    pickup_datetime|   dropoff_datetime|passenger_count|trip_time_in_secs|trip_distance|pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|pickup_borough|dropoff_borough|
+--------------------+--------------------+---------+---------+------------------+-------------------+-------------------+---------------+-----------------+-------------+----------------+---------------+-----------------+----------------+--------------+---------------+
|89D227B655E5C82AE...|BA96DE419E711691B...|      CMT|        1|                 N|2013-01-01 15:11:48|2013-01-01 15:18:10|              4|              382|          1.0|      -73.978165|   

In [24]:
# Droping the rows where the borough can't be defined as it will not benefit us in the analysis
taxiDF = taxiDF.filter(
    col("pickup_borough").isNotNull() & col("dropoff_borough").isNotNull()
)

In [25]:
# Droping the outlier in the trip_time_in_secs 
taxiDF = taxiDF.filter(
    (col("trip_time_in_secs") > 0) & (col("trip_time_in_secs") <= 4 * 3600)
)

In [26]:
taxiDF = taxiDF.withColumn('pickup_ts', taxiDF['pickup_datetime'].cast("long"))\
                .withColumn('dropoff_ts', taxiDF['dropoff_datetime'].cast("long"))

In [27]:
# Defining the window to use it in the lag window function to know the previous dropoff_ts
windowSpec = Window \
    .partitionBy(taxiDF['medallion'])\
    .orderBy(taxiDF['pickup_ts'].asc())

In [28]:
taxiDF = taxiDF.withColumn('previous_dropoff_ts', lag(taxiDF['dropoff_ts']).over(windowSpec))

In [29]:
taxiDF = taxiDF.withColumn('idle_time', taxiDF['pickup_ts'] - taxiDF['previous_dropoff_ts'])

In [34]:
taxiDF.show(2)

+--------------------+--------------------+---------+---------+------------------+-------------------+-------------------+---------------+-----------------+-------------+----------------+---------------+-----------------+----------------+--------------+---------------+----------+----------+-------------------+---------+---------------+
|           medallion|        hack_license|vendor_id|rate_code|store_and_fwd_flag|    pickup_datetime|   dropoff_datetime|passenger_count|trip_time_in_secs|trip_distance|pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|pickup_borough|dropoff_borough| pickup_ts|dropoff_ts|previous_dropoff_ts|idle_time|idle_time_clean|
+--------------------+--------------------+---------+---------+------------------+-------------------+-------------------+---------------+-----------------+-------------+----------------+---------------+-----------------+----------------+--------------+---------------+----------+----------+-------------------+---------+---

In [33]:
# Filtering the idle_time to exclude the rows where idle_time > 4 hours

taxiDF = taxiDF.withColumn(
    "idle_time_clean",
    when(col("idle_time") < 4 * 3600, col("idle_time")).otherwise(None)
)

In [35]:
taxiDF.select(
    avg("idle_time").alias('AVG_idle_time')
).show()

+-----------------+
|    AVG_idle_time|
+-----------------+
|2471.618618552623|
+-----------------+



In [36]:
taxiDF.filter(taxiDF.pickup_borough == taxiDF.dropoff_borough).count()

85945

In [37]:
taxiDF.filter(taxiDF.pickup_borough != taxiDF.dropoff_borough).count()

11431

In [38]:
result = taxiDF.groupBy("dropoff_borough").agg(
    avg("idle_time").alias("avg_idle_time"),
    max("idle_time").alias("max_idle_time"),
    count("*").alias("trip_count")
).show()

+---------------+-----------------+-------------+----------+
|dropoff_borough|    avg_idle_time|max_idle_time|trip_count|
+---------------+-----------------+-------------+----------+
|       Brooklyn|5366.267429760666|       932220|      3582|
|  Staten Island|           6390.0|        21720|        12|
|      Manhattan|2206.220800019266|      1547693|     87959|
|         Queens|5058.991154170177|      1034280|      5430|
|          Bronx|6790.402476780186|       870240|       393|
+---------------+-----------------+-------------+----------+



In [48]:
result = taxiDF.groupBy("medallion").agg(
    (sum("trip_time_in_secs").alias("sum_trip_time_in_secs") / \
    (sum(coalesce("idle_time", lit(0))).alias("sum_idle_time") + \
     sum("trip_time_in_secs").alias("sum_trip_time_in_secs"))).alias('utilization')
).orderBy(col("utilization").desc())\
.show(5)

+--------------------+-----------+
|           medallion|utilization|
+--------------------+-----------+
|11483B90AA9E57FFD...|        1.0|
|1216C3718738FF4A9...|        1.0|
|02BFD1B64C2C80B43...|        1.0|
|1A30D2B12B2E61370...|        1.0|
|0C58E359F3F8CDF2D...|        1.0|
+--------------------+-----------+
only showing top 5 rows
